In [ ]:
# LANGGRAPH + AZURE OPENAI — MEDICAL QUERY ROUTING

from typing import Annotated, TypedDict
import operator
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import AzureChatOpenAI

# Azure OpenAI
llm = AzureChatOpenAI(
    azure_endpoint="https://YOUR-RESOURCE.openai.azure.com/",
    api_key="YOUR_API_KEY", api_version="2024-10-21",
    azure_deployment="gpt-4.1", temperature=0
)

# Tools
def web_search(query: str) -> str:
    return f"Web Search Result: {query}"

def pubmed_search(query: str) -> str:
    return f"PubMed Result: {query}"

# State
class AgentState(TypedDict):
    question: str
    tool: str
    answer: str
    history: Annotated[list[dict], operator.add]

# Router prompt
tool_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an intelligent healthcare routing agent.
Medical/health/disease/symptoms/drugs/treatment/diagnosis/clinical
→ pubmed. Everything else → web. Reply ONLY: pubmed OR web."""),
    ("human", "{question}")
])

VALID_TOOLS = {"pubmed", "web"}

# Router node
def decide_tool(state: AgentState) -> dict:
    chain = tool_prompt | llm
    tool = chain.invoke(
        {"question": state["question"]}
    ).content.strip().lower()
    tool = tool if tool in VALID_TOOLS else "web"
    print(f"Selected Tool: {tool}")
    return {"tool": tool}

# Tool execution node
def execute_tool(state: AgentState) -> dict:
    q, tool = state["question"], state.get("tool", "web")
    result = pubmed_search(q) if tool == "pubmed" else web_search(q)
    return {
        "answer": result,
        "history": [{"question": q, "tool": tool, "answer": result}]
    }

# Build Graph
builder = StateGraph(AgentState)
builder.add_node("decide_tool", decide_tool)
builder.add_node("execute_tool", execute_tool)
builder.add_edge(START, "decide_tool")
builder.add_edge("decide_tool", "execute_tool")
builder.add_edge("execute_tool", END)
graph = builder.compile()

# Invoke
result = graph.invoke({
    "question": "What is the treatment for diabetes?",
    "tool": "", "answer": "", "history": []
})

print(f"\nAnswer: {result['answer']}")
print(f"History: {result['history']}")